In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv('momo.csv')
df.columns

In [ ]:
momo_features = df2.groupby("nameOrig").agg(
    transaction_count=("amount", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
    max_amount=("amount", "max"),
    min_amount=("amount", "min"),
    std_amount=("amount", "std"),
    avg_oldbalance=("oldbalanceOrg", "mean"),
    avg_newbalance=("newbalanceOrig", "mean"),
    unique_destinations=("nameDest", "nunique"),
    avg_step=("step", "mean"),
    std_step=("step", "std")
).reset_index()

In [ ]:
#fill missing values 
momo_features = momo_features.fillna(0)

In [ ]:
#Add extra features 
momo_features["amount_range"] = momo_features["max_amount"] - momo_features["min_amount"]
momo_features["amount_per_transaction"] = momo_features["total_amount"] / (momo_features["transaction_count"] + 1)
momo_features["destination_ratio"] = momo_features["unique_destinations"] / (momo_features["transaction_count"] + 1)

In [ ]:
# preparing training features
X_momo = momo_features.drop("nameOrig", axis=1)

In [ ]:
#s caling faetures
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_momo_scaled = scaler.fit_transform(X_momo)

In [ ]:
# Train isolajtion forest
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)

iso_model.fit(X_momo_scaled)

In [ ]:
#Get anomaly outputs
anomaly_scores = iso_model.decision_function(X_momo_scaled)
behavior_risk = -anomaly_scores

In [ ]:
#Normalize to 0 1
risk_scaler = MinMaxScaler()
behavior_risk_normalized = risk_scaler.fit_transform(
    behavior_risk.reshape(-1, 1)
).flatten()

In [ ]:
#save results
momo_features["behavior_risk"] = behavior_risk_normalized

In [ ]:
#inspect
momo_features["behavior_risk"].describe()

In [ ]:
#simulate equal length (demo purpose)
min_len = min(len(default_probs), len(behavior_risk_normalized))

final_risk = (
    0.7 * default_probs[:min_len] +
    0.3 * behavior_risk_normalized[:min_len]
)

final_risk[:10]

In [ ]:
import joblib

joblib.dump(model_a, "model_a.pkl")
joblib.dump(iso_model, "iso_model_b.pkl")
joblib.dump(scaler, "scaler_b.pkl")
joblib.dump(risk_scaler, "risk_scaler_b.pkl")